# conv-padding-zero composite — cx7: conv1d with both zero padding and stride > 1

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `conv-padding-zero`, `conv-stride-downsample`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "conv-padding-zero"
DD_ATOM_IDS = ["conv-padding-zero", "conv-stride-downsample"]
DD_SUBTOPICS = ["CNN: Conv zero padding", "CNN: Stride downsample arithmetic"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA's from-scratch `conv1d` separates the conv into two prep steps before the einsum:
1. **Zero-pad** the input on both sides with `padding=P` zeros (the `conv-padding-zero` atom). After this, the effective input length becomes `W + 2*P`.
2. **Stride-downsample** when extracting windows: the number of windows over the padded input with stride `S` is `OW = (W + 2*P - K) // S + 1` (the `conv-stride-downsample` atom — note the `+1` for the leading window).

The composition: pad first, THEN apply the strided output-length formula on top of the padded length. This drill exercises both in one function: pad the input, return the padded tensor AND the predicted strided output length.

**Anatomy.**
- `x_pad = x.new_zeros(B, IC, W + 2*P); x_pad[..., P:P+W] = x` — atom A.
- `OW = (W + 2*P - K) // S + 1` — atom B applied to the PADDED length.

**Why this matters.** ResNet-style 'stride-2 + same-padding' downsampling relies on this exact composition: the same-padding term cancels the floor-division by-one error so the output is *exactly* `W // S`. Forget the `2*P` term and you get the canonical off-by-one bug.

### Composite Exercise — conv1d with both zero padding and stride > 1

**Atoms exercised together**: `conv-padding-zero`, `conv-stride-downsample`

Implement `cx7_pad_then_strided_outlen(x, K, S, P)`.

- `x`: float tensor of shape `(B, IC, W)`.
- `K`: kernel width (int).
- `S`: stride (int, >= 1).
- `P`: padding (int, >= 0). Same amount on left and right.

Return `(x_padded, OW)`:
- `x_padded`: shape `(B, IC, W + 2*P)`. Interior `[P : P+W]` equals `x`; left/right `P` columns are exactly zero.
- `OW`: integer output width of a stride-`S` conv with kernel `K` over the padded input.

1. **Pad** — allocate a zero buffer of shape `(B, IC, W + 2*P)` via `x.new_zeros(...)` so the dtype/device track `x`. Slice-assign `x` into columns `[P : P+W]`.
2. **Strided output length** — apply `OW = (W + 2*P - K) // S + 1` to the PADDED length. Note this includes the `+1` for the leading window.

In [ ]:
def cx7_pad_then_strided_outlen(x, K, S, P):
    B, IC, W = x.shape
    # Atom A (conv-padding-zero): allocate a zero buffer, slice-assign the interior.
    x_pad = x.new_zeros(B, IC, W + 2 * P)
    x_pad[..., P : P + W] = x
    # Atom B (conv-stride-downsample): formula applied to the PADDED length.
    # Note the +1 — leading window starts at index 0.
    OW = (W + 2 * P - K) // S + 1
    return x_pad, OW


<details><summary>Show solution — cx7</summary>

```python
def cx7_pad_then_strided_outlen(x, K, S, P):
    B, IC, W = x.shape
    # Atom A (conv-padding-zero): allocate a zero buffer, slice-assign the interior.
    x_pad = x.new_zeros(B, IC, W + 2 * P)
    x_pad[..., P : P + W] = x
    # Atom B (conv-stride-downsample): formula applied to the PADDED length.
    # Note the +1 — leading window starts at index 0.
    OW = (W + 2 * P - K) // S + 1
    return x_pad, OW
```

The two atoms compose in series: pad expands the effective input by `2*P`, then the strided output formula consumes that expanded length. Forgetting the `+1` or the `2*P` term is the canonical conv-shape bug. The same-pad + stride-2 case (Case C) is why ResNets work cleanly: with `K=3, P=1, S=2` the formula collapses to `W // 2` for even `W`.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx7',
        'subtopics': ["CNN: Conv zero padding", "CNN: Stride downsample arithmetic"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()